# POC 02 — Incremental API ingestion
Attach `lh_api_orders` as the default Lakehouse. Upload the first two JSON pages before run 1, then add page 3 before run 2.

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType,
    DecimalType, ArrayType, BooleanType)

source_path = 'Files/api_source/*.json'
event_schema = StructType([
    StructField('event_id', StringType(), True),
    StructField('order_id', StringType(), True),
    StructField('customer_id', StringType(), True),
    StructField('status', StringType(), True),
    StructField('amount', DecimalType(12, 2), True),
    StructField('updated_at', StringType(), True),
])

api_schema = StructType([
    # Real API envelope: one row per response page.
    StructField('data', ArrayType(event_schema), True),
    StructField('pagination', StructType([
        StructField('count', IntegerType(), True),
        StructField('has_more', BooleanType(), True),
        StructField('next_cursor', StringType(), True),
    ]), True),
    StructField('request', StructType([
        StructField('updated_after', StringType(), True),
        StructField('available_through', StringType(), True),
    ]), True),
    # Flat JSON Lines from the downloadable mock-data path.
    StructField('event_id', StringType(), True),
    StructField('page_number', IntegerType(), True),
    StructField('order_id', StringType(), True),
    StructField('customer_id', StringType(), True),
    StructField('status', StringType(), True),
    StructField('amount', DecimalType(12, 2), True),
    StructField('updated_at', StringType(), True),
    StructField('_corrupt_record', StringType(), True),
])

incoming_raw = (spark.read
    .schema(api_schema)
    .option('mode', 'PERMISSIVE')
    .option('columnNameOfCorruptRecord', '_corrupt_record')
    .json(source_path)
    .withColumn('_source_file', F.input_file_name())
    .cache())
incoming_raw.count()  # Materialize once so corrupt-record diagnostics are queryable.

malformed = incoming_raw.filter(F.col('_corrupt_record').isNotNull())
if malformed.count():
    display(malformed.select('_source_file', '_corrupt_record'))
    raise ValueError('Malformed JSON found. Correct or remove the file shown above.')

# Explode data[] from real API pages; keep compatibility with flat mock JSON rows.
envelope_events = (incoming_raw
    .filter(F.col('data').isNotNull())
    .select(F.explode('data').alias('event'), '_source_file')
    .select(
        'event.event_id', F.lit(None).cast('int').alias('page_number'),
        'event.order_id', 'event.customer_id', 'event.status',
        'event.amount', 'event.updated_at', '_source_file'))

flat_events = (incoming_raw
    .filter(F.col('data').isNull())
    .select('event_id', 'page_number', 'order_id', 'customer_id',
        'status', 'amount', 'updated_at', '_source_file'))

incoming_events = envelope_events.unionByName(flat_events)
incoming_parsed = incoming_events.withColumn(
    '_parsed_updated_at',
    F.coalesce(
        F.to_timestamp('updated_at', "yyyy-MM-dd'T'HH:mm:ssX"),
        F.to_timestamp('updated_at')))

invalid = incoming_parsed.filter(
    F.col('event_id').isNull()
    | F.col('order_id').isNull()
    | F.col('_parsed_updated_at').isNull())
invalid_count = invalid.count()

if invalid_count:
    print(f'Invalid API rows: {invalid_count}. Inspect the file and raw values below.')
    display(invalid.select(
        '_source_file', 'event_id', 'order_id',
        'updated_at', '_parsed_updated_at'))
    raise ValueError('Invalid API rows found. Remove or correct the file shown above, then rerun this cell.')

incoming = (incoming_parsed
    .select(
        'event_id', 'page_number', 'order_id', 'customer_id',
        'status', 'amount',
        F.col('_parsed_updated_at').alias('updated_at'),
        '_source_file')
    .withColumn('_ingested_at', F.current_timestamp())
    .dropDuplicates(['event_id']))
display(incoming)

## Bronze — idempotent event landing
`event_id` is immutable. MERGE inserts unseen events and ignores pages already processed.

In [ ]:
table_name = 'bronze_api_order_events'
before_count = spark.table(table_name).count() if spark.catalog.tableExists(table_name) else 0

if spark.catalog.tableExists(table_name):
    (DeltaTable.forName(spark, table_name).alias('target')
        .merge(incoming.alias('source'), 'target.event_id = source.event_id')
        .whenNotMatchedInsertAll()
        .execute())
else:
    incoming.write.format('delta').mode('overwrite').saveAsTable(table_name)

bronze = spark.table(table_name)
after_count = bronze.count()
new_events = after_count - before_count
print(f'New events inserted: {new_events}; Bronze events: {after_count}')

## Silver — current state per business key
An order may have many events. Silver keeps only the newest `updated_at` for each `order_id`.

In [ ]:
latest = Window.partitionBy('order_id').orderBy(F.col('updated_at').desc(), F.col('event_id').desc())
silver = (bronze
    .withColumn('_version_rank', F.row_number().over(latest))
    .filter(F.col('_version_rank') == 1)
    .drop('_version_rank'))
silver.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('silver_api_orders_current')
display(silver.orderBy('order_id'))

## Gold, watermark, and run evidence

In [ ]:
gold = (silver.groupBy('status')
    .agg(F.countDistinct('order_id').alias('orders_count'),
         F.sum('amount').alias('total_amount')))
gold.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('gold_api_order_summary')

watermark = bronze.agg(F.max('updated_at')).first()[0]
(spark.createDataFrame([(watermark,)], ['last_successful_updated_at'])
    .write.format('delta').mode('overwrite').saveAsTable('control_api_watermark'))

run_log = (spark.range(1).select(
    F.current_timestamp().alias('run_at'),
    F.lit(incoming_events.count()).alias('source_rows_scanned'),
    F.lit(new_events).alias('new_events_inserted'),
    F.lit(after_count).alias('bronze_events_after')))
run_log.write.format('delta').mode('append').saveAsTable('api_ingestion_run_log')
display(gold.orderBy('status'))